# Final test results: half B

**CRISP-DM stage:** Evaluation

Every configuration was scored once on half B, with the settings and both lines fixed in the Lock (DECISIONS.md, 17 Nov 2026). Nothing in this notebook chooses anything: it only applies the locked lines and the locked analysis plan.

1. **Repeat check.** Each Final test run scored half A again first. If any half A number differs from the locked one, the notebook stops.
2. **Headline table on half B** with 95 percent bootstrap intervals for image F1 and image AU-ROC.
3. **Costs at the locked lines**, at the main business setting and over the sensitivity ranges.
4. **Paired differences**, for the locked pairs.
5. **Secondary metrics** for the appendix.
6. **Figures.**

**Careful when reading:** half B has only 2 good parts per product, so the intervals are wide. Pixel metrics are uncertain, because the masks were drawn by people and their agreement cannot be measured here. If half B disagrees with half A, it is reported as it is, and the locked choice stays.

**Runs on:** the laptop, from the saved photo scores and metrics.

In [ ]:
import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Finds the project folder, whether the notebook runs from the project or from notebooks/
PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "src").exists():
    PROJECT_FOLDER = PROJECT_FOLDER.parent
sys.path.insert(0, str(PROJECT_FOLDER))
from src import cost
from src import evaluate

RESULTS = PROJECT_FOLDER / "outputs/results"
FINAL_FOLDER = RESULTS / "final_test"
HALF_B_FOLDER = FINAL_FOLDER / "half_b"
RERUN_FOLDER = FINAL_FOLDER / "half_a_rerun"
FIGURE_FOLDER = PROJECT_FOLDER / "outputs/figures"
# Where the locked half A files are
LOCKED_FOLDERS = [RESULTS / "05_core", RESULTS / "07_core", RESULTS / "08_resolution"]
PRODUCTS = ["sheet_metal", "vial"]
RUN_IDS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C6_512"]
NAMES = {"C1": "C1 PaDiM, ResNet-18", "C2": "C2 PaDiM, Wide ResNet-50", "C3": "C3 PaDiM, EfficientNet-B0",
         "C4": "C4 PatchCore, ResNet-18", "C5": "C5 PatchCore, Wide ResNet-50",
         "C6": "C6 PatchCore, EfficientNet-B0", "C7": "C7 WinCLIP, CLIP",
         "C8": "C8 Reverse Distillation, Wide ResNet-50", "C6_512": "C6 at 512 pixels"}
# The same colour for each configuration as in the Phase 8 chart
COLOURS = {"C1": "#2a78d6", "C2": "#eb6834", "C3": "#56b4e9", "C4": "#1baf7a", "C5": "#eda100",
           "C6": "#cc79a7", "C7": "#7a5195", "C8": "#8c6d31", "C6_512": "#cc79a7"}
FIGURE_DPI = 150


def seed_of(file_name):
    """Return the seed from a name such as C5_vial_seed43_halfB_metrics.json, here 43."""
    return file_name.split("_seed")[1].split("_")[0]


def read_scores(score_file):
    """Return file names, labels (True for defective) and scores from one run's file."""
    names = []
    labels = []
    scores = []
    with open(score_file, encoding="utf-8") as file:
        for row in csv.DictReader(file):
            names.append(row["file_name"])
            labels.append(row["label"] == "bad")
            scores.append(float(row["score"]))
    return names, torch.tensor(labels), torch.tensor(scores)


def mean_and_spread(values):
    """Return the mean and the spread (half the distance from lowest to highest) of a list."""
    mean = sum(values) / len(values)
    spread = (max(values) - min(values)) / 2
    return mean, spread

## 1. Repeat check: does half A come out exactly as locked?

Every Final test run scored half A before half B. Its numbers are compared with the locked half A files. **If any number differs, the notebook stops here**, as the Lock requires.

In [ ]:
COMPARED = ["threshold", "au_pro_5_percent", "au_pro_30_percent", "pixel_f1", "image_f1", "image_au_roc",
            "good_validation_photos_flagged", "good_half_a_photos_flagged", "defective_half_a_photos_flagged"]
repeat_rows = []
for rerun_file in sorted(RERUN_FOLDER.rglob("*_metrics.json")):
    locked_file = None
    for folder in LOCKED_FOLDERS:
        for candidate in folder.rglob(rerun_file.name):
            locked_file = candidate
    if locked_file is None:
        raise FileNotFoundError("No locked half A file for " + rerun_file.name)
    with open(rerun_file, encoding="utf-8") as file:
        rerun = json.load(file)
    with open(locked_file, encoding="utf-8") as file:
        locked = json.load(file)
    differences = []
    for key in COMPARED:
        if rerun[key] != locked[key]:
            differences.append(key)
    repeat_rows.append({"run": rerun_file.name.replace("_metrics.json", ""),
                        "repeats_exactly": len(differences) == 0,
                        "differences": " ".join(differences)})

with open(FINAL_FOLDER / "half_a_repeat_check.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["run", "repeats_exactly", "differences"])
    writer.writeheader()
    for row in repeat_rows:
        writer.writerow(row)
failed = []
for row in repeat_rows:
    if not row["repeats_exactly"]:
        failed.append(row["run"] + ": " + row["differences"])
print(len(repeat_rows), "runs checked,", len(failed), "differ")
if len(failed) > 0:
    raise ValueError("Half A did not repeat, stop and report: " + "; ".join(failed))

## 2. Read the half B runs and the locked lines

For each configuration and product: every seed's half B photo scores and metrics, the locked cost-based line of that seed (Lock table), and the locked half A numbers for comparison.

In [ ]:
# Locked cost-based lines: every business setting for C1 to C8, the main setting for C6 at 512
locked_lines = {}
with open(RESULTS / "07_evaluation/cost_by_run.csv", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        key = (row["config"], row["product"], row["seed"], float(row["defect_rate"]), float(row["cost_ratio"]))
        locked_lines[key] = float(row["cheapest_threshold"])
with open(RESULTS / "08_resolution/C6_512_thresholds.csv", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        key = (row["config"], row["product"], row["seed"], cost.MAIN_DEFECT_RATE, float(cost.MAIN_COST_RATIO))
        locked_lines[key] = float(row["cost_based_threshold"])

runs = {}
for run_id in RUN_IDS:
    for product in PRODUCTS:
        metric_files = sorted((HALF_B_FOLDER / run_id).glob(run_id + "_" + product + "_seed*_halfB_metrics.json"))
        if len(metric_files) == 0:
            raise FileNotFoundError("No half B files for " + run_id + " " + product)
        run = {"seeds": [], "metrics": [], "scores": [], "benchmark_lines": [], "cost_lines": [],
               "names": None, "labels": None, "half_a": []}
        for metric_file in metric_files:
            seed = seed_of(metric_file.name)
            with open(metric_file, encoding="utf-8") as file:
                metrics = json.load(file)
            names, labels, scores = read_scores(Path(str(metric_file).replace("_metrics.json", "_photo_scores.csv")))
            # Every seed and every configuration must list the same photos in the same order
            if run["names"] is None:
                run["names"] = names
                run["labels"] = labels
            elif names != run["names"]:
                raise ValueError("Photo order differs within " + run_id + " " + product)
            run["seeds"].append(seed)
            run["metrics"].append(metrics)
            run["scores"].append(scores)
            run["benchmark_lines"].append(metrics["threshold"])
            run["cost_lines"].append(locked_lines[(run_id, product, seed, cost.MAIN_DEFECT_RATE, float(cost.MAIN_COST_RATIO))])
            for folder in LOCKED_FOLDERS:
                for locked_file in folder.rglob(run_id + "_" + product + "_seed" + seed + "_metrics.json"):
                    with open(locked_file, encoding="utf-8") as file:
                        run["half_a"].append(json.load(file))
        runs[(run_id, product)] = run

for product in PRODUCTS:
    first = runs[("C1", product)]
    for run_id in RUN_IDS:
        if runs[(run_id, product)]["names"] != first["names"]:
            raise ValueError("Photo order differs between C1 and " + run_id + " on " + product)
    print(product, ":", len(first["names"]), "half B photos,", int(first["labels"].sum()), "defective")
print(len(runs), "configuration and product pairs read")

## 3. Headline table on half B, with bootstrap intervals

The bootstrap is the locked one from `src/evaluate.py`: 2,000 draws with seed 42. Each draw takes parts with repeats, good and defective separately, so their numbers stay the same, and the interval is the middle 95 percent. For PatchCore, each draw averages the 3 seeds. Image F1 is shown at both lines. The half A AU-ROC is shown beside half B, to see whether half A predicted half B.

In [ ]:
HEADLINE = ["au_pro_5_percent", "au_pro_30_percent", "pixel_f1", "image_f1", "image_au_roc",
            "seconds_per_photo", "peak_gpu_gb"]
table_rows = []
for product in PRODUCTS:
    for run_id in RUN_IDS:
        run = runs[(run_id, product)]
        row = {"product": product, "config": run_id, "seeds": len(run["seeds"])}
        for key in HEADLINE:
            values = []
            for metrics in run["metrics"]:
                values.append(metrics[key])
            mean, spread = mean_and_spread(values)
            row[key] = round(mean, 4)
            row[key + "_spread"] = round(spread, 4)
        # Image F1 at the locked cost-based line
        cost_f1 = []
        for seed_number in range(len(run["seeds"])):
            flagged = run["scores"][seed_number] > run["cost_lines"][seed_number]
            cost_f1.append(evaluate.f1_from_flags(flagged, run["labels"]))
        row["image_f1_cost_line"] = round(sum(cost_f1) / len(cost_f1), 4)
        share_defective = float(run["labels"].float().mean())
        row["flag_everything_f1"] = round(2 * share_defective / (1 + share_defective), 4)
        intervals = evaluate.bootstrap_image_metrics(run["scores"], run["benchmark_lines"], run["labels"], run["names"])
        row["image_f1_low"] = round(intervals["image_f1_low"], 4)
        row["image_f1_high"] = round(intervals["image_f1_high"], 4)
        row["image_au_roc_low"] = round(intervals["image_au_roc_low"], 4)
        row["image_au_roc_high"] = round(intervals["image_au_roc_high"], 4)
        cost_intervals = evaluate.bootstrap_image_metrics(run["scores"], run["cost_lines"], run["labels"], run["names"])
        row["image_f1_cost_line_low"] = round(cost_intervals["image_f1_low"], 4)
        row["image_f1_cost_line_high"] = round(cost_intervals["image_f1_high"], 4)
        row["good_parts"] = intervals["good_parts"]
        row["defective_parts"] = intervals["defective_parts"]
        half_a_auroc = []
        for metrics in run["half_a"]:
            half_a_auroc.append(metrics["image_au_roc"])
        row["half_a_image_au_roc"] = round(sum(half_a_auroc) / len(half_a_auroc), 4)
        table_rows.append(row)
        print(product.ljust(12), run_id.ljust(7), "AU-ROC", row["image_au_roc"],
              "[" + str(row["image_au_roc_low"]) + " to " + str(row["image_au_roc_high"]) + "]",
              "(half A " + str(row["half_a_image_au_roc"]) + ")",
              "| image F1 benchmark", row["image_f1"], "cost line", row["image_f1_cost_line"],
              "| AU-PRO 30%", row["au_pro_30_percent"], "| pixel F1", row["pixel_f1"])

with open(FINAL_FOLDER / "final_results_table.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(table_rows[0].keys()))
    writer.writeheader()
    for row in table_rows:
        writer.writerow(row)


def with_spread(row, key):
    """Write a value, with its spread in brackets when there are several seeds."""
    if row["seeds"] > 1:
        return str(round(row[key], 3)) + " (" + str(round(row[key + "_spread"], 3)) + ")"
    return str(round(row[key], 3))


lines = ["# Final test results on half B", "",
         "Settings and lines were locked on half A before half B was opened (DECISIONS.md, 17 Nov 2026). "
         "Each value is the mean over seeds, with the spread in brackets for 3 seeds. "
         "Intervals are 95 percent part bootstraps.", ""]
for product in PRODUCTS:
    lines.append("## " + product)
    lines.append("")
    lines.append("| Configuration | Seeds | AU-PRO 5% | AU-PRO 30% | Pixel F1 | Image F1, benchmark line [interval] | "
                 "Image F1, cost-based line [interval] | Flag-everything F1 | Image AU-ROC [interval] | Half A AU-ROC | "
                 "Seconds per photo | Peak GPU GB |")
    lines.append("| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |")
    for row in table_rows:
        if row["product"] != product:
            continue
        lines.append("| " + NAMES[row["config"]] + " | " + str(row["seeds"]) + " | " + with_spread(row, "au_pro_5_percent")
                     + " | " + with_spread(row, "au_pro_30_percent") + " | " + with_spread(row, "pixel_f1")
                     + " | " + with_spread(row, "image_f1") + " [" + str(round(row["image_f1_low"], 3)) + " to "
                     + str(round(row["image_f1_high"], 3)) + "] | " + str(round(row["image_f1_cost_line"], 3)) + " ["
                     + str(round(row["image_f1_cost_line_low"], 3)) + " to " + str(round(row["image_f1_cost_line_high"], 3))
                     + "] | " + str(row["flag_everything_f1"]) + " | " + with_spread(row, "image_au_roc") + " ["
                     + str(round(row["image_au_roc_low"], 3)) + " to " + str(round(row["image_au_roc_high"], 3)) + "] | "
                     + str(round(row["half_a_image_au_roc"], 3)) + " | " + with_spread(row, "seconds_per_photo")
                     + " | " + with_spread(row, "peak_gpu_gb") + " |")
    lines.append("")
    good_parts = table_rows[0]["good_parts"]
    lines.append("Intervals come from drawing the " + str(good_parts) + " good and the defective parts of half B, "
                 "2,000 times. With so few good parts, they are wide and rough.")
    lines.append("")
lines.append("Pixel metrics are uncertain, because masks were drawn by people and their agreement cannot be measured here.")
with open(FINAL_FOLDER / "final_results_table.md", "w", encoding="utf-8") as file:
    file.write("\n".join(lines) + "\n")
print("Saved the table")

## 4. Costs at the locked lines

The expected cost per part, `defect rate x miss rate x cost ratio + (1 - defect rate) x false-reject rate`, from `src/cost.py`. **Main setting:** 1 percent defects, a miss costs 10 false rejects. Passing everything costs 0.1 per part at this setting, so a method is only useful below 0.1. **Sensitivity:** each setting's own half A line is applied to half B unchanged.

In [ ]:
cost_rows = []
for product in PRODUCTS:
    for run_id in RUN_IDS:
        run = runs[(run_id, product)]
        totals = {"cost_line": [], "cost_line_miss": [], "cost_line_false_reject": [],
                  "benchmark": [], "benchmark_miss": [], "benchmark_false_reject": [],
                  "precision_cost_line": [], "recall_cost_line": [], "precision_benchmark": [], "recall_benchmark": []}
        for seed_number in range(len(run["seeds"])):
            scores = run["scores"][seed_number]
            at_cost = cost.cost_at(scores, run["labels"], run["cost_lines"][seed_number], cost.MAIN_DEFECT_RATE, cost.MAIN_COST_RATIO)
            at_bench = cost.cost_at(scores, run["labels"], run["benchmark_lines"][seed_number], cost.MAIN_DEFECT_RATE, cost.MAIN_COST_RATIO)
            totals["cost_line"].append(at_cost["cost"])
            totals["cost_line_miss"].append(at_cost["miss_rate"])
            totals["cost_line_false_reject"].append(at_cost["false_reject_rate"])
            totals["benchmark"].append(at_bench["cost"])
            totals["benchmark_miss"].append(at_bench["miss_rate"])
            totals["benchmark_false_reject"].append(at_bench["false_reject_rate"])
            counts = evaluate.precision_recall_at(scores, run["labels"], run["cost_lines"][seed_number])
            totals["precision_cost_line"].append(counts["precision"])
            totals["recall_cost_line"].append(counts["recall"])
            counts = evaluate.precision_recall_at(scores, run["labels"], run["benchmark_lines"][seed_number])
            totals["precision_benchmark"].append(counts["precision"])
            totals["recall_benchmark"].append(counts["recall"])
        row = {"product": product, "config": run_id}
        for key in totals:
            values = totals[key]
            if "not defined" in values:
                row[key] = "not defined"
            else:
                row[key] = round(sum(values) / len(values), 4)
        row["pass_everything"] = cost.MAIN_DEFECT_RATE * cost.MAIN_COST_RATIO
        cost_rows.append(row)
        print(product.ljust(12), run_id.ljust(7), "cost-based line: cost", row["cost_line"], "miss", row["cost_line_miss"],
              "false reject", row["cost_line_false_reject"], "| benchmark line: cost", row["benchmark"])
with open(FINAL_FOLDER / "final_cost_main_setting.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(cost_rows[0].keys()))
    writer.writeheader()
    for row in cost_rows:
        writer.writerow(row)

# Sensitivity: C1 to C8 only, because C6 at 512 has a locked line for the main setting only
sensitivity_rows = []
for product in PRODUCTS:
    for run_id in RUN_IDS[:8]:
        run = runs[(run_id, product)]
        for defect_rate in cost.DEFECT_RATES:
            for cost_ratio in cost.COST_RATIOS:
                values = []
                for seed_number in range(len(run["seeds"])):
                    line = locked_lines[(run_id, product, run["seeds"][seed_number], float(defect_rate), float(cost_ratio))]
                    values.append(cost.cost_at(run["scores"][seed_number], run["labels"], line, defect_rate, cost_ratio)["cost"])
                sensitivity_rows.append({"product": product, "config": run_id, "defect_rate": defect_rate,
                                         "cost_ratio": cost_ratio, "cost": round(sum(values) / len(values), 4),
                                         "pass_everything": round(defect_rate * cost_ratio, 4)})
with open(FINAL_FOLDER / "final_cost_sensitivity.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(sensitivity_rows[0].keys()))
    writer.writeheader()
    for row in sensitivity_rows:
        writer.writerow(row)

# Which configuration is cheapest at each setting, as a plain count
for product in PRODUCTS:
    winners = {}
    for defect_rate in cost.DEFECT_RATES:
        for cost_ratio in cost.COST_RATIOS:
            best = None
            for row in sensitivity_rows:
                if row["product"] == product and row["defect_rate"] == defect_rate and row["cost_ratio"] == cost_ratio:
                    if best is None or row["cost"] < best["cost"]:
                        best = row
            if best["config"] not in winners:
                winners[best["config"]] = 0
            winners[best["config"]] = winners[best["config"]] + 1
    print(product, "cheapest at the locked lines, out of 18 settings:", winners)

## 5. Paired differences on half B

The locked pairs: the Phase 5 and Phase 7 comparisons, C6 at 512 against 256, and each recommended configuration against its runner-up (C6 against C5 on Sheet Metal and C7 against C5 on Vial, both already in the Phase 7 list). Image F1 uses the benchmark line, as in Phases 5 and 7. A difference counts as real only if its interval leaves out 0.

In [ ]:
PAIRS = [("C4", "C1", "method, ResNet-18: PatchCore against PaDiM"),
         ("C5", "C2", "method, Wide ResNet-50: PatchCore against PaDiM"),
         ("C2", "C1", "network, PaDiM: Wide ResNet-50 against ResNet-18"),
         ("C5", "C4", "network, PatchCore: Wide ResNet-50 against ResNet-18"),
         ("C3", "C1", "network, PaDiM: EfficientNet-B0 against ResNet-18"),
         ("C6", "C4", "network, PatchCore: EfficientNet-B0 against ResNet-18"),
         ("C6", "C5", "network, PatchCore: EfficientNet-B0 against Wide ResNet-50"),
         ("C8", "C5", "method, Wide ResNet-50: Reverse Distillation against PatchCore"),
         ("C7", "C5", "method: WinCLIP against PatchCore with Wide ResNet-50"),
         ("C6_512", "C6", "resolution: C6 at 512 against 256")]
paired_rows = []
for product in PRODUCTS:
    for first_id, second_id, comparison in PAIRS:
        first = runs[(first_id, product)]
        second = runs[(second_id, product)]
        gap = evaluate.paired_bootstrap_difference((first["scores"], first["benchmark_lines"]),
                                                   (second["scores"], second["benchmark_lines"]),
                                                   first["labels"], first["names"])
        row = {"product": product, "comparison": comparison, "first_minus_second": first_id + " minus " + second_id}
        for key in gap:
            row[key] = round(gap[key], 4)
        row["image_f1_real"] = gap["image_f1_difference_low"] > 0 or gap["image_f1_difference_high"] < 0
        row["image_au_roc_real"] = gap["image_au_roc_difference_low"] > 0 or gap["image_au_roc_difference_high"] < 0
        paired_rows.append(row)
        print(product.ljust(12), (first_id + " - " + second_id).ljust(14),
              "AU-ROC", row["image_au_roc_difference"], "[" + str(row["image_au_roc_difference_low"]) + " to "
              + str(row["image_au_roc_difference_high"]) + "]", "real" if row["image_au_roc_real"] else "not real",
              "| F1", row["image_f1_difference"], "real" if row["image_f1_real"] else "not real")
with open(FINAL_FOLDER / "final_paired_differences.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(paired_rows[0].keys()))
    writer.writeheader()
    for row in paired_rows:
        writer.writerow(row)

## 6. Secondary metrics for the appendix

Image AUPRC must be read against the random-guess level, which is the share of defective photos. Pixel average precision comes from the heat maps, measured on Kaggle.

In [ ]:
secondary_rows = []
for product in PRODUCTS:
    for run_id in RUN_IDS:
        run = runs[(run_id, product)]
        auprc = []
        pixel_ap = []
        for metrics in run["metrics"]:
            auprc.append(metrics["image_auprc"])
            pixel_ap.append(metrics["pixel_average_precision"])
        cost_row = None
        for row in cost_rows:
            if row["product"] == product and row["config"] == run_id:
                cost_row = row
        secondary_rows.append({"product": product, "config": run_id,
                               "random_guess_auprc": round(float(run["labels"].float().mean()), 4),
                               "image_auprc": round(sum(auprc) / len(auprc), 4),
                               "pixel_average_precision": round(sum(pixel_ap) / len(pixel_ap), 4),
                               "benchmark_precision": cost_row["precision_benchmark"],
                               "benchmark_recall": cost_row["recall_benchmark"],
                               "cost_line_precision": cost_row["precision_cost_line"],
                               "cost_line_recall": cost_row["recall_cost_line"]})
with open(FINAL_FOLDER / "final_secondary_metrics.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(secondary_rows[0].keys()))
    writer.writeheader()
    for row in secondary_rows:
        writer.writerow(row)
for row in secondary_rows:
    print(row)

## 7. Figures

- **Half A against half B AU-ROC:** points on the diagonal mean half A predicted half B well. The bars are half B's intervals.
- **Cost at the locked lines:** the main setting, with passing everything as the dashed line.
- **Score histograms and confusion matrices** for the recommended combination of each product (C6 on Sheet Metal, seed 42; C7 on Vial).

In [ ]:
# Figure 1: half A against half B image AU-ROC
figure, axes = plt.subplots(1, 2, figsize=(11, 5.2))
for column in range(len(PRODUCTS)):
    product = PRODUCTS[column]
    axis = axes[column]
    axis.plot([0, 1], [0, 1], color="#999999", linestyle="--", linewidth=1)
    panel_rows = []
    for row in table_rows:
        if row["product"] == product:
            panel_rows.append(row)

    def half_b_value(row):
        return row["image_au_roc"]

    # Labels sorted by height and pushed at least 0.045 apart, so they never overlap
    panel_rows.sort(key=half_b_value)
    label_heights = {}
    last_height = None
    for row in panel_rows:
        height = row["image_au_roc"]
        if last_height is not None and height < last_height + 0.045:
            height = last_height + 0.045
        label_heights[row["config"]] = height
        last_height = height
    for row in panel_rows:
        x = row["half_a_image_au_roc"]
        y = row["image_au_roc"]
        marker = "D" if row["config"] == "C6_512" else "o"
        axis.errorbar([x], [y], yerr=[[y - row["image_au_roc_low"]], [row["image_au_roc_high"] - y]],
                      fmt=marker, color=COLOURS[row["config"]], ecolor=COLOURS[row["config"]], elinewidth=1,
                      capsize=3, markersize=8)
        label = "C6 at 512" if row["config"] == "C6_512" else row["config"]
        # The label sits to the right of the right-most point, joined to its own point by a thin line
        axis.annotate(label, (x, y), xytext=(1.045, label_heights[row["config"]]), textcoords="data",
                      fontsize=9, color="#333333", va="center",
                      arrowprops={"arrowstyle": "-", "color": COLOURS[row["config"]], "lw": 0.6})
    axis.set_xlim(0.3, 1.2)
    axis.set_ylim(0.0, 1.02)
    axis.set_xlabel("Half A image AU-ROC (development)")
    axis.set_ylabel("Half B image AU-ROC (final), with 95% interval")
    axis.set_title(product.replace("_", " ").title())
    axis.grid(alpha=0.3)
figure.suptitle("Did half A predict half B? Dashed line: the same on both halves")
figure.tight_layout()
figure.savefig(FIGURE_FOLDER / "final_half_a_vs_half_b_auroc.png", dpi=FIGURE_DPI)
plt.close(figure)

# Figure 2: cost per part at the main setting, both locked lines
figure, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True)
for column in range(len(PRODUCTS)):
    product = PRODUCTS[column]
    axis = axes[column]
    labels = []
    cost_values = []
    benchmark_values = []
    for row in cost_rows:
        if row["product"] == product:
            labels.append("C6 at 512" if row["config"] == "C6_512" else row["config"])
            cost_values.append(row["cost_line"])
            benchmark_values.append(min(row["benchmark"], 1.0))
    positions = list(range(len(labels)))
    upper = []
    lower = []
    for position in positions:
        upper.append(position - 0.2)
        lower.append(position + 0.2)
    axis.barh(upper, cost_values, height=0.38, color="#2a78d6", label="cost-based line (locked)")
    axis.barh(lower, benchmark_values, height=0.38, color="#bbbbbb", label="benchmark line")
    axis.axvline(cost.MAIN_DEFECT_RATE * cost.MAIN_COST_RATIO, color="#333333", linestyle="--", linewidth=1)
    axis.text(cost.MAIN_DEFECT_RATE * cost.MAIN_COST_RATIO + 0.01, len(labels) - 0.5, "pass everything", fontsize=8)
    axis.set_yticks(positions)
    axis.set_yticklabels(labels)
    axis.invert_yaxis()
    axis.set_xlabel("Expected cost per part (1% defects, a miss costs 10)")
    axis.set_title(product.replace("_", " ").title())
    axis.grid(axis="x", alpha=0.3)
    if column == 0:
        axis.legend(loc="lower right", fontsize=8)
figure.suptitle("Half B cost at the locked lines. Lower is better")
figure.tight_layout()
figure.savefig(FIGURE_FOLDER / "final_cost_at_locked_lines.png", dpi=FIGURE_DPI)
plt.close(figure)

# Figures 3 and 4: the recommended combination per product
RECOMMENDED = {"sheet_metal": "C6", "vial": "C7"}
figure, axes = plt.subplots(2, 2, figsize=(11, 8))
for column in range(len(PRODUCTS)):
    product = PRODUCTS[column]
    run_id = RECOMMENDED[product]
    run = runs[(run_id, product)]
    # The first seed, so the histogram shows one real model and not an average
    scores = run["scores"][0]
    labels = run["labels"]
    good_scores = scores[~labels].tolist()
    defective_scores = scores[labels].tolist()
    axis = axes[0][column]
    axis.hist(good_scores, bins=15, color="#1baf7a", alpha=0.7, label="good")
    axis.hist(defective_scores, bins=15, color="#eb6834", alpha=0.7, label="defective")
    axis.axvline(run["cost_lines"][0], color="#2a78d6", linewidth=2, label="cost-based line")
    axis.axvline(run["benchmark_lines"][0], color="#333333", linestyle="--", linewidth=1.5, label="benchmark line")
    axis.set_title(product.replace("_", " ").title() + ", " + NAMES[run_id] + ", seed " + run["seeds"][0])
    axis.set_xlabel("Photo score")
    axis.set_ylabel("Half B photos")
    axis.legend(fontsize=8)
    counts = cost.confusion_counts(scores, labels, run["cost_lines"][0])
    axis = axes[1][column]
    grid = [[counts["correct_passes"], counts["false_rejects"]], [counts["missed_defects"], counts["correct_rejects"]]]
    axis.imshow(grid, cmap="Blues")
    for row_number in range(2):
        for column_number in range(2):
            axis.text(column_number, row_number, str(grid[row_number][column_number]), ha="center", va="center", fontsize=14)
    axis.set_xticks([0, 1])
    axis.set_xticklabels(["passed", "rejected"])
    axis.set_yticks([0, 1])
    axis.set_yticklabels(["good", "defective"])
    axis.set_title("At the locked cost-based line")
figure.tight_layout()
figure.savefig(FIGURE_FOLDER / "final_recommended_scores_and_confusion.png", dpi=FIGURE_DPI)
plt.close(figure)
print("Saved 3 figures")